## Load the summary information and preprocessed data

In [1]:
import os
import json
import pandas as pd

from pathlib import Path

from single_sample_gsea import ss_gsea


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 10 samples and 367 columns.
First few rows of the data:


,progression-free-survival_days,recurrence,overall-survival_days,survival,sample_title,tissue,tumor type,Sex,ISG15,TNFRSF18,...,TLR7,TLR8,CYBB,FOXP3,XAGE1A,XAGE1B,IL2RG,SH2D1A,CD40LG,G6PD
0,144,1,234,0,D01,lung,adenocarcinoma,female,8.882173,6.171445,...,5.284179,4.930614,8.576640,6.549989,8.074424,8.069476,7.577147,5.239719,5.509732,6.336172
1,1088,0,1088,0,D03,lung,adenocarcinoma,male,7.529397,5.248833,...,2.258170,3.079047,7.128186,3.958780,8.727017,8.725554,5.703426,3.967348,3.158742,6.494435
2,253,1,262,1,D06,lung,Large cell neuroendocarine carcinoma,male,8.251118,4.840282,...,5.837569,4.775874,8.362806,4.759907,0.000000,0.000000,7.162295,5.826552,3.886130,6.795918
3,259,1,259,1,D07,lung,adenocarcinoma,female,7.544768,4.033201,...,5.105420,3.636555,8.202394,4.115164,8.507414,8.501555,6.781775,5.287543,4.261330,6.541916
4,191,1,376,1,D08,lung,Large cell neuroendocarine carcinoma,male,7.112337,4.998744,...,6.330000,4.266616,8.244475,5.164976,7.329526,7.330164,6.668930,4.644956,3.693069,9.176531


## Load gene sets

In [2]:
MSIGDB_JSON = Path("../../resources/h.all.v2026.1.Hs.json")

with MSIGDB_JSON.open(encoding="utf-8") as f:
    gene_sets = {name: data["geneSymbols"] for name, data in json.load(f).items()}

print(f"Loaded {len(gene_sets)} gene sets.")


Loaded 50 gene sets.


## Compute single sample GSEA on the preprocessed data

Select only gene columns for the computation.

In [3]:
# IMPORTANT: This is dataset-specific and is the only variable that needs to be changed for each of the dataset.
sample_id_column = 'sample_title'

gene_expression_df = df.iloc[:, summary_data['data_summary']['clinical_features']:].copy()
gene_expression_df.index = df[sample_id_column].astype(str)
gene_expression_df = gene_expression_df.T
gene_expression_df.index.name = "gene"

result = ss_gsea(gene_expression_df, gene_sets)


/Users/jakakokosar/dev/phd/survival-rnaseq-data/datasets/.venv/lib/python3.14/site-packages/single_sample_gsea/ssgsea.py:51: RuntimeWarning: invalid value encountered in divide
  signature_ecdf = signature_ecdf / np.sum(signature_ecdf, axis=0)


## Merge the results with the clinical data

The output table should have the same number of rows as before and columns should sum up to clinical data + 50 gene sets.

In [4]:
clinical_df = df.iloc[:, :summary_data["data_summary"]["clinical_features"]].copy()
clinical_df = clinical_df.set_index(sample_id_column)

output_df = pd.concat([clinical_df, result], axis=1)
output_df = output_df.reset_index(drop=False)

output_df.to_csv(f"{GEO_ID}_preprocessed_ssgsea.csv", index=False)

# sanity check (there is 50 gene sets)
assert output_df.shape[1] == summary_data["data_summary"]["clinical_features"] + 50
